# Actividad de clase: 
**- Segmentación y propiedades de regiones**

**Reto:** identificar objetos verdes sobre un fondo, contarlos y describir su tamaño, ubicación y forma. Primero obtendrá una máscara binaria (foreground = objeto, background = fondo). Después usará `skimage.measure.label` y `regionprops_table` para medir cada región.

**Entregables:** notebook ejecutado en formato pdf, tabla de regiones, figura con objetos numerados y respuestas a las preguntas. Requiere `numpy`, `matplotlib`, `opencv-python`, `pandas` y `scikit-image` (instalación: `pip install scikit-image`).

---

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import filters, measure, morphology, segmentation

## 1. Generación de imagen de sintética

Observe qué regiones deberían ser objetos y cuántas espera detectar. El objeto pequeño a la derecha permite discutir la elección de un área mínima.


In [ ]:
RUTA = 'prueba.tif'

Generación de la imagen

In [ ]:
rng = np.random.default_rng(4)
rgb = np.full((280, 400, 3), (205, 190, 170), dtype=np.uint8)
cv2.ellipse(rgb, (95, 105), (65, 35), -25, 0, 360, (55, 145, 70), -1)
cv2.ellipse(rgb, (275, 100), (72, 36), 35, 0, 360, (65, 160, 75), -1)
cv2.ellipse(rgb, (185, 218), (55, 28), -10, 0, 360, (50, 125, 65), -1)
cv2.circle(rgb, (365, 220), 5, (50, 150, 65), -1)
rgb = np.clip(rgb.astype(float) + rng.normal(0, 6, rgb.shape), 0, 255).astype(np.uint8)

Visualización

In [ ]:
plt.imshow(rgb)
plt.title('Imagen de entrada')
plt.axis('off')
plt.show()

In [ ]:
# salvar la imagen
cv2.imwrite(RUTA, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

## 2. Segmentación del primer plano y fondo

Proponga un algoritmo de segmentación que genere una mascaras binaria, llamada `segmentation`, cuyo pixeles tienen los siguientes valores: `foreground = 1` y `background = 0`.


In [ ]:
# --> aquí su código

segmentation = False #< -- asigne la mascara a esta variable !!!

## 3. Componentes conectados

Cada grupo conectado de píxeles del `foreground` recibe una etiqueta distinta; el fondo conserva la etiqueta 0.

In [ ]:
etiquetas, n = measure.label(segmentation, connectivity=2, return_num=True)
print(f"Número de regiones: {n}")

Visualización de la segmentación

In [ ]:
plt.figure()
plt.title("Etiquetas de regiones")
plt.imshow(L, cmap='nipy_spectral')
plt.colorbar()
plt.show()

## 4. Medir regiones con `regionprops_table`

El área se expresa en el número de píxeles y el perímetro en píxeles. El centroide y la caja delimitadora usan coordenadas **fila, columna**. La excentricidad se acerca a 0 en regiones circulares y a 1 en regiones alargadas; la solidez es la razón entre el área y el área de la envolvente convexa.


In [ ]:
gris = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

propiedades = measure.regionprops_table( etiquetas, intensity_image=gris,
    properties=('label', 'area', 'centroid', 'bbox', 'perimeter', 'eccentricity', 'solidity', 'mean_intensity'))

# Convertir las propiedades a un DataFrame para facilitar la visualización
tabla_df = pd.DataFrame(propiedades)
print("Propiedades de las regiones:")
display(tabla_df)

## 5. Localizar los objetos

Muestre la imagen original con el contorno rojo de cada región detectada y su número de etiqueta ubicado cerca del centroide y el área en pixeles.  Se descartan grupos con menos de `AREA_MINIMA` píxeles. Modifique ese valor y compruebe si desaparece algún objeto que sí desea conservar.

In [ ]:
# Aquí su código
AREA_MINIMA = 0